# Parsing backend comparison

Use this workspace to run both CAMAT parsing backends on the same files and inspect any differences in the generated dataframes.


In [1]:
# Helpers and imports

from typing import Callable, Dict, List, Optional

import pandas as pd
from IPython.display import display

from py_scripts.music21_backend import parse_files as parse_files_music21
from py_scripts.partitura_backend import parse_files_partitura

_PARSER_MAP: Dict[str, Callable] = {
    "music21": parse_files_music21,
    "partitura": parse_files_partitura,
}

_COMPARISON_COLUMN_ORDER: List[str] = [
    "Measure",
    "Global Onset",
    "Local Onset",
    "Duration",
    "MIDI",
    "Pitch",
    "Voice",
]


def _preferred_column_order(columns: List[str]) -> List[str]:
    primary = [col for col in _COMPARISON_COLUMN_ORDER if col in columns]
    trailing = [col for col in columns if col not in primary]
    return primary + trailing


def normalize_dataframe(df: pd.DataFrame, decimals: int = 6) -> pd.DataFrame:
    """Return a stable, sorted copy of the dataframe for backend comparisons."""
    df_norm = df.copy()
    numeric_cols = list(df_norm.select_dtypes(include="number").columns)
    if numeric_cols:
        df_norm[numeric_cols] = (
            df_norm[numeric_cols]
            .apply(pd.to_numeric, errors="coerce")
            .round(decimals)
        )
    ordered_cols = _preferred_column_order(list(df_norm.columns))
    if ordered_cols:
        df_norm = df_norm[ordered_cols]
        df_norm = df_norm.sort_values(ordered_cols).reset_index(drop=True)
    return df_norm


def row_difference_table(df_a: pd.DataFrame, df_b: pd.DataFrame) -> pd.DataFrame:
    """Aggregate unmatched rows between two normalized dataframes."""
    if df_a.empty and df_b.empty:
        return pd.DataFrame(columns=list(df_a.columns) + ["music21_count", "partitura_count", "delta"])
    grouped_a = (
        df_a.groupby(list(df_a.columns), dropna=False)
        .size()
        .rename("music21_count")
    )
    grouped_b = (
        df_b.groupby(list(df_b.columns), dropna=False)
        .size()
        .rename("partitura_count")
    )
    combined = pd.concat([grouped_a, grouped_b], axis=1).fillna(0).astype(int)
    combined["delta"] = combined["music21_count"] - combined["partitura_count"]
    diff = combined[combined["delta"] != 0]
    if diff.empty:
        return pd.DataFrame(columns=list(df_a.columns) + ["music21_count", "partitura_count", "delta"])
    diff = diff.reset_index()
    return diff


def _expand_difference_rows(diff_df: pd.DataFrame, data_columns: List[str], *, copies_from: str) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for _, row in diff_df.iterrows():
        copies = int(abs(row[copies_from]))
        if copies <= 0:
            continue
        base = {col: row[col] for col in data_columns}
        rows.extend([base] * copies)
    if rows:
        return pd.DataFrame(rows, columns=data_columns)
    return pd.DataFrame(columns=data_columns)


def compare_dataframes(
    df_music21: pd.DataFrame,
    df_partitura: pd.DataFrame,
    *,
    decimals: int = 6,
    ignore_columns: Optional[List[str]] = None,
) -> Dict[str, object]:
    """Compare two backend outputs and return a structured diff."""
    ignore_columns = ignore_columns or []

    df_music21_cmp = df_music21.drop(columns=[col for col in ignore_columns if col in df_music21.columns], errors="ignore")
    df_partitura_cmp = df_partitura.drop(columns=[col for col in ignore_columns if col in df_partitura.columns], errors="ignore")

    norm_music21 = normalize_dataframe(df_music21_cmp, decimals=decimals)
    norm_partitura = normalize_dataframe(df_partitura_cmp, decimals=decimals)

    common_cols = [col for col in norm_music21.columns if col in norm_partitura.columns]
    music21_only_cols = [col for col in norm_music21.columns if col not in common_cols]
    partitura_only_cols = [col for col in norm_partitura.columns if col not in common_cols]

    row_diffs = pd.DataFrame()
    cell_diffs = None
    side_by_side_diffs = pd.DataFrame()
    music21_only_summary = pd.DataFrame()
    partitura_only_summary = pd.DataFrame()
    music21_only_rows = pd.DataFrame()
    partitura_only_rows = pd.DataFrame()
    frames_equal = False

    if common_cols:
        cmp_music21 = norm_music21[common_cols].sort_values(common_cols).reset_index(drop=True)
        cmp_partitura = norm_partitura[common_cols].sort_values(common_cols).reset_index(drop=True)
        frames_equal = (
            not music21_only_cols
            and not partitura_only_cols
            and cmp_music21.equals(cmp_partitura)
        )
        row_diffs = row_difference_table(cmp_music21, cmp_partitura)

        if cmp_music21.shape == cmp_partitura.shape and not frames_equal:
            cell_diffs = cmp_music21.compare(cmp_partitura, align_axis=0)
            if cell_diffs is not None and not cell_diffs.empty:
                side_by_side_diffs = pd.concat(
                    [cmp_music21.add_suffix("_music21"), cmp_partitura.add_suffix("_partitura")],
                    axis=1,
                )
                diff_mask = cmp_music21.ne(cmp_partitura)
                side_by_side_diffs = side_by_side_diffs.loc[diff_mask.any(axis=1)].reset_index(drop=True)

        if not row_diffs.empty:
            data_columns = [col for col in row_diffs.columns if col not in {"music21_count", "partitura_count", "delta"}]
            music21_only_summary = row_diffs[row_diffs["delta"] > 0].reset_index(drop=True)
            partitura_only_summary = row_diffs[row_diffs["delta"] < 0].reset_index(drop=True)
            if data_columns:
                if not music21_only_summary.empty:
                    music21_only_rows = _expand_difference_rows(music21_only_summary.assign(delta=lambda df: df["delta"].abs()), data_columns, copies_from="delta")
                if not partitura_only_summary.empty:
                    partitura_only_rows = _expand_difference_rows(partitura_only_summary.assign(delta=lambda df: df["delta"].abs()), data_columns, copies_from="delta")

    return {
        "frames_equal": frames_equal,
        "row_delta": len(df_music21) - len(df_partitura),
        "row_differences": row_diffs,
        "cell_differences": cell_diffs,
        "side_by_side_differences": side_by_side_diffs,
        "common_columns": common_cols,
        "music21_only_columns": music21_only_cols,
        "partitura_only_columns": partitura_only_cols,
        "normalized_music21": norm_music21,
        "normalized_partitura": norm_partitura,
        "music21_only_summary": music21_only_summary,
        "partitura_only_summary": partitura_only_summary,
        "music21_only_rows": music21_only_rows,
        "partitura_only_rows": partitura_only_rows,
    }


In [2]:
# Configuration

FILE_SOURCES = [
    'https://analyse.hfm-weimar.de/database/02/PrJode_Jos0302_COM_1-5_MissaDapac_002_00006.xml',
    'https://raw.githubusercontent.com/humdrum-tools/bach-wtc-fugues/refs/heads/master/kern/wtc1f04.krn',
    'https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei',
    'https://raw.githubusercontent.com/humdrum-tools/bach-wtc/refs/heads/main/kern/wtc2f05.krn',
    'C:/Users/egorp/OneDrive/Desktop/weimar_ftp_backup/dokuwiki/database/05/CaSe_11_UNSP_UNSP_Danksagenw_005_00011.xml',
]

FILTER_ZERO_DURATION = True
ADJUST_FRACTIONAL_DURATION = True
STRIP_TIES = True  # music21 backend only

IGNORE_COLUMNS = ["Voice"]  # drop these columns before comparing (set [] to compare everything)

DISPLAY_PREVIEW = False
PREVIEW_ROWS = 20

PLOT_BACKEND = "none"  # "plt" | "bokeh" | "none"
SHOW_MEASURE_LINES = True
RETURN_PLOTS = False

PLOT_SIZE_X = 900
PLOT_SIZE_Y = 600

ZOOM_DRAG_DIM = "both"
ZOOM_WHEEL_DIM = "width"

CLEANUP_REMOTE = True
SHOW_PROGRESS = False

COMPARISON_DECIMALS = 6


In [3]:
# Parse each backend

comparison_outputs: Dict[str, Dict[str, object]] = {}
for backend, parser_fn in _PARSER_MAP.items():
    print(f"Running {backend} backend...")
    try:
        extra_kwargs = {"strip_ties": STRIP_TIES} if backend == "music21" else {}
        results, dfs_by_name, _ = parser_fn(
            FILE_SOURCES,
            filter_zero_duration=FILTER_ZERO_DURATION,
            adjust_fractional_duration=ADJUST_FRACTIONAL_DURATION,
            backend=PLOT_BACKEND,
            show_measure_lines=SHOW_MEASURE_LINES,
            display_preview=DISPLAY_PREVIEW,
            preview_rows=PREVIEW_ROWS,
            cleanup_remote=CLEANUP_REMOTE,
            return_plots=RETURN_PLOTS,
            plot_width=PLOT_SIZE_X,
            plot_height=PLOT_SIZE_Y,
            zoom_drag_dim=ZOOM_DRAG_DIM,
            zoom_wheel_dim=ZOOM_WHEEL_DIM,
            show_progress=SHOW_PROGRESS,
            progress_desc=f"Parsing ({backend})",
            **extra_kwargs,
        )
    except Exception as exc:
        print(f"{backend} backend failed: {exc}")
        results, dfs_by_name = [], {}
    comparison_outputs[backend] = {
        "results": results,
        "dfs_by_name": dfs_by_name,
    }
print("Done.")


Running music21 backend...
Processing (music21): PrJode_Jos0302_COM_1-5_MissaDapac_002_00006.xml -> 00_prjode_jos0302_com_1_5_missadapac_002_00006
Processing (music21): wtc1f04.krn -> 01_wtc1f04
Processing (music21): Bach-JS_Ein_feste_Burg.mei -> 02_bach_js_ein_feste_burg
Processing (music21): wtc2f05.krn -> 03_wtc2f05
Processing (music21): CaSe_11_UNSP_UNSP_Danksagenw_005_00011.xml -> 04_case_11_unsp_unsp_danksagenw_005_00011
Running partitura backend...
Processing (partitura): PrJode_Jos0302_COM_1-5_MissaDapac_002_00006.xml -> 00_prjode_jos0302_com_1_5_missadapac_002_00006
Processing (partitura): wtc1f04.krn -> 01_wtc1f04


c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importkern.py:295: UserWarning: Input line 836 contained no data and will not be counted towards `max_rows=50000`.  This differs from the behaviour in NumPy <=1.22 which counted lines rather than rows.  If desired, the previous behaviour can be achieved by using `itertools.islice`.
Please see the 1.23 release notes for an example on how to do this.  If you wish to ignore this warning, use `warnings.filterwarnings`.  This warning is expected to be removed in the future and is given only once per `loadtxt` call.
  file = np.loadtxt(
c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importkern.py:355: UserWarning: Part P0 already exists. Adding to previous Part.
  warnings.warn(


Processing (partitura): Bach-JS_Ein_feste_Burg.mei -> 02_bach_js_ein_feste_burg


c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importmei.py:255: UserWarning: The key signature is not encoded in None or in any ancestor scoreDef.
  warnings.warn(
c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importmei.py:258: UserWarning: A default key signature of C maj is set.
  warnings.warn("A default key signature of C maj is set.")


Processing (partitura): wtc2f05.krn -> 03_wtc2f05
Processing (partitura): CaSe_11_UNSP_UNSP_Danksagenw_005_00011.xml -> 04_case_11_unsp_unsp_danksagenw_005_00011
Done.


c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\directions.py:514: UserWarning: error parsing "None" (TypeError)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))


In [4]:
# Summaries per backend

summary_rows = []
for backend, data in comparison_outputs.items():
    for item in data.get("results", []):
        df = item.get("df")
        if df is None:
            continue
        pos_duration = df.loc[df["Duration"] > 0, "Duration"] if "Duration" in df.columns else pd.Series(dtype=float)
        min_duration = float(pos_duration.min()) if not pos_duration.empty else float("nan")
        max_global = float(df["Global Onset"].max()) if "Global Onset" in df.columns and not df.empty else float("nan")
        summary_rows.append({
            "backend": backend,
            "name": item.get("name"),
            "rows": len(df),
            "unique_midi": df["MIDI"].nunique() if "MIDI" in df.columns else float("nan"),
            "min_duration": min_duration,
            "max_global_onset": max_global,
        })
if summary_rows:
    summary_df = pd.DataFrame(summary_rows).sort_values(["name", "backend"]).reset_index(drop=True)
    display(summary_df)
else:
    print("No parsed data to summarise.")


,backend,name,rows,unique_midi,min_duration,max_global_onset
0,music21,00_prjode_jos0302_com_1_5_missadapac_002_00006,755,24,0.50,668.0
1,partitura,00_prjode_jos0302_com_1_5_missadapac_002_00006,755,24,0.50,668.0
2,music21,01_wtc1f04,1326,45,0.50,456.0
3,partitura,01_wtc1f04,1326,45,0.50,456.0
4,music21,02_bach_js_ein_feste_burg,235,25,0.25,47.0
5,partitura,02_bach_js_ein_feste_burg,235,25,0.25,46.0
6,music21,03_wtc2f05,880,39,0.25,198.0
7,partitura,03_wtc2f05,880,39,0.25,198.0
8,music21,04_case_11_unsp_unsp_danksagenw_005_00011,268,19,2.00,236.0
9,partitura,04_case_11_unsp_unsp_danksagenw_005_00011,268,19,2.00,236.0


In [5]:
# Detailed diffs

diffs_by_file: Dict[str, Dict[str, object]] = {}
music21_dfs = comparison_outputs.get("music21", {}).get("dfs_by_name", {})
partitura_dfs = comparison_outputs.get("partitura", {}).get("dfs_by_name", {})
all_names = sorted(set(music21_dfs) | set(partitura_dfs))
if not all_names:
    print("No files to compare.")
else:
    for name in all_names:
        df_music21 = music21_dfs.get(name)
        df_partitura = partitura_dfs.get(name)
        print(f"=== {name} ===")
        if df_music21 is None:
            message = "Missing dataframe for music21 backend."
            print(message)
            diffs_by_file[name] = {"error": message}
            continue
        if df_partitura is None:
            message = "Missing dataframe for partitura backend."
            print(message)
            diffs_by_file[name] = {"error": message}
            continue
        comparison = compare_dataframes(
            df_music21,
            df_partitura,
            decimals=COMPARISON_DECIMALS,
            ignore_columns=IGNORE_COLUMNS,
        )
        diffs_by_file[name] = comparison
        print(f"Frames equal (normalised): {comparison['frames_equal']}")
        print(f"Row delta (music21 - partitura): {comparison['row_delta']}")
        if comparison["music21_only_columns"]:
            print(f"Columns only in music21: {comparison['music21_only_columns']}")
        if comparison["partitura_only_columns"]:
            print(f"Columns only in partitura: {comparison['partitura_only_columns']}")

        row_diffs = comparison["row_differences"]
        if row_diffs.empty:
            print("No unmatched rows after normalization.")
        else:
            print("Aggregated row differences:")
            display(row_diffs.head(20))

        music21_only_summary = comparison["music21_only_summary"]
        if not music21_only_summary.empty:
            print("Rows only in music21 (aggregated counts):")
            display(music21_only_summary)
            music21_only_rows = comparison["music21_only_rows"]
            if not music21_only_rows.empty:
                print("Expanded rows (music21 only):")
                display(music21_only_rows.head(20))

        partitura_only_summary = comparison["partitura_only_summary"]
        if not partitura_only_summary.empty:
            print("Rows only in partitura (aggregated counts):")
            display(partitura_only_summary)
            partitura_only_rows = comparison["partitura_only_rows"]
            if not partitura_only_rows.empty:
                print("Expanded rows (partitura only):")
                display(partitura_only_rows.head(20))

        side_by_side = comparison["side_by_side_differences"]
        if side_by_side is not None and not side_by_side.empty:
            print("Cell-level differences (side-by-side):")
            display(side_by_side.head(20))
        print()


=== 00_prjode_jos0302_com_1_5_missadapac_002_00006 ===
Frames equal (normalised): False
Row delta (music21 - partitura): 0
Aggregated row differences:


,Measure,Global Onset,Local Onset,Duration,MIDI,Pitch,music21_count,partitura_count,delta
0,2,12.0,0.0,3.0,55,G3,1,0,1
1,2,12.0,0.0,8.0,67,G4,1,0,1
2,2,15.0,3.0,1.0,57,A3,1,0,1
3,2,16.0,4.0,2.0,58,B-3,1,0,1
4,2,18.0,6.0,2.0,55,G3,1,0,1
5,2,20.0,8.0,4.0,57,A3,1,0,1
6,2,20.0,8.0,4.0,65,F4,1,0,1
7,3,24.0,0.0,1.0,55,G3,1,0,1
8,3,24.0,0.0,3.0,67,G4,1,0,1
9,3,24.0,0.0,8.0,55,G3,1,0,1


Rows only in music21 (aggregated counts):


,Measure,Global Onset,Local Onset,Duration,MIDI,Pitch,music21_count,partitura_count,delta
0,2,12.0,0.0,3.0,55,G3,1,0,1
1,2,12.0,0.0,8.0,67,G4,1,0,1
2,2,15.0,3.0,1.0,57,A3,1,0,1
3,2,16.0,4.0,2.0,58,B-3,1,0,1
4,2,18.0,6.0,2.0,55,G3,1,0,1
...,...,...,...,...,...,...,...,...,...
747,63,666.0,10.0,1.0,66,F#4,1,0,1
748,63,667.0,11.0,1.0,64,E4,1,0,1
749,64,668.0,0.0,12.0,43,G2,1,0,1
750,64,668.0,0.0,12.0,55,G3,1,0,1


Expanded rows (music21 only):


...

,Measure,Global Onset,Local Onset,Duration,MIDI,Pitch,music21_count,partitura_count,delta
0,1,4.0,4.0,2.0,48,C3,0,1,-1
1,1,4.0,4.0,2.0,60,C4,0,1,-1
2,1,4.0,4.0,2.0,64,E4,0,1,-1
3,1,4.0,4.0,2.0,67,G4,0,1,-1
4,1,6.0,6.0,2.0,55,G3,0,1,-1
...,...,...,...,...,...,...,...,...,...
252,1,228.0,228.0,4.0,69,A4,0,1,-1
253,1,232.0,232.0,4.0,50,D3,0,1,-1
254,1,232.0,232.0,4.0,59,B3,0,1,-1
255,1,232.0,232.0,4.0,67,G4,0,2,-2


Expanded rows (partitura only):


,Measure,Global Onset,Local Onset,Duration,MIDI,Pitch
0,1,4.0,4.0,2.0,48,C3
1,1,4.0,4.0,2.0,60,C4
2,1,4.0,4.0,2.0,64,E4
3,1,4.0,4.0,2.0,67,G4
4,1,6.0,6.0,2.0,55,G3
5,1,6.0,6.0,2.0,59,B3
6,1,6.0,6.0,2.0,62,D4
7,1,6.0,6.0,2.0,67,G4
8,1,8.0,8.0,4.0,53,F3
9,1,8.0,8.0,4.0,60,C4


Cell-level differences (side-by-side):


,Measure_music21,Global Onset_music21,Local Onset_music21,Duration_music21,MIDI_music21,Pitch_music21,Measure_partitura,Global Onset_partitura,Local Onset_partitura,Duration_partitura,MIDI_partitura,Pitch_partitura
0,2,4.0,0.0,2.0,48,C3,1,4.0,4.0,2.0,48,C3
1,2,4.0,0.0,2.0,60,C4,1,4.0,4.0,2.0,60,C4
2,2,4.0,0.0,2.0,64,E4,1,4.0,4.0,2.0,64,E4
3,2,4.0,0.0,2.0,67,G4,1,4.0,4.0,2.0,67,G4
4,2,6.0,2.0,2.0,55,G3,1,6.0,6.0,2.0,55,G3
5,2,6.0,2.0,2.0,59,B3,1,6.0,6.0,2.0,59,B3
6,2,6.0,2.0,2.0,62,D4,1,6.0,6.0,2.0,62,D4
7,2,6.0,2.0,2.0,67,G4,1,6.0,6.0,2.0,67,G4
8,3,8.0,0.0,4.0,53,F3,1,8.0,8.0,4.0,53,F3
9,3,8.0,0.0,4.0,60,C4,1,8.0,8.0,4.0,60,C4
